# Pipeline 3D — nuages de points, mesh, volume

**Prérequis :** exécuter `depth_field_V3_2d.ipynb` jusqu’à la fin pour générer le fichier :

`data/processed/depth_field_v3_bundle.npz`

Ce notebook charge ce bundle (profondeurs + masque + RGB rognés) puis enchaîne les visualisations Open3D et le calcul de volume approché.


In [1]:
# Chargement du bundle produit par depth_field_V3_2d.ipynb
from pathlib import Path
import numpy as np
from PIL import Image
import open3d as o3d

EXPORT_PATH = Path("C:/Users/mvm/open3d_vision/data/processed") / "depth_field_v3_bundle.npz"
if not EXPORT_PATH.is_file():
    raise FileNotFoundError(
        "Fichier introuvable : exécuter d'abord depth_field_V3_2d.ipynb (cellule d'export).\n"
        f"Attendu : {EXPORT_PATH}"
    )

z = np.load(EXPORT_PATH)
depth_1 = z["depth_1"].astype(np.float64)
depth_2_aligned = z["depth_2_remapped"].astype(np.float64)
mask_combined = z["mask_combined"].astype(np.uint8)
rgb1 = z["rgb1"].astype(np.uint8)
rgb2 = z["rgb2"].astype(np.uint8)
pad = int(z["pad"])

image_cropped_1 = Image.fromarray(rgb1)
image_cropped_2 = Image.fromarray(rgb2)

h_d1, w_d1 = depth_1.shape
depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)
if depth_2_aligned.shape != (h_d1, w_d1):
    import cv2
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w_d1, h_d1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)

depth_1_masked = np.where(mask_combined == 255, depth_1, np.nan)
depth_2_masked = np.where(mask_combined == 255, depth_2_aligned, np.nan)

print(f"Chargé : {EXPORT_PATH} — depth {h_d1}x{w_d1}, pad={pad}")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Chargé : C:\Users\mvm\open3d_vision\data\processed\depth_field_v3_bundle.npz — depth 1811x1350, pad=16


In [2]:
# Fonctions utilitaires: Inverser les normales de mesh_super_1 et mesh_super_2 (via fonctions)


def _flip_mesh_normals(mesh):
    """Retourne un mesh avec normales inversées (sommets + faces)."""
    m = o3d.geometry.TriangleMesh(mesh)
    # Inversion robuste: inverse les orientations des triangles
    tris = np.asarray(m.triangles)
    if tris.size > 0:
        m.triangles = o3d.utility.Vector3iVector(tris[:, [0, 2, 1]])
    m.compute_triangle_normals()
    m.compute_vertex_normals()
    return m


def _flip_both_mesh_normals(mesh_a, mesh_b):
    """Inversion des normales pour deux meshes."""
    return _flip_mesh_normals(mesh_a), _flip_mesh_normals(mesh_b)



In [3]:
#Fonctions utilitaires : affichage de normales directement dans un mesh 
def _lineset_vertex_normals(mesh, scale=None, max_segments=8000):
    """Segments jaunes = normales aux sommets (sous-échantillonnées)."""
    v = np.asarray(mesh.vertices, dtype=np.float64)
    n = np.asarray(mesh.vertex_normals, dtype=np.float64)
    if v.size == 0 or n.size == 0:
        return None
    step = max(1, int(np.ceil(len(v) / max_segments)))
    v = v[::step]
    n = n[::step]
    nn = np.linalg.norm(n, axis=1, keepdims=True)
    n = np.divide(n, np.maximum(nn, 1e-12))
    if scale is None:
        ext = np.linalg.norm(np.asarray(mesh.get_axis_aligned_bounding_box().get_extent()))
        scale = 0.015 * float(ext)
    ends = v + scale * n
    pts = np.vstack([v, ends])
    m = len(v)
    lines = np.array([[i, i + m] for i in range(m)], dtype=np.int32)
    ls = o3d.geometry.LineSet()
    ls.points = o3d.utility.Vector3dVector(pts)
    ls.lines = o3d.utility.Vector2iVector(lines)
    ls.paint_uniform_color([1.0, 0.85, 0.15])
    return ls


### Étapes 3D (Open3D)

Les variables `depth_1_masked`, `depth_2_masked`, `image_cropped_1/2`, `mask_combined` viennent du chargement ci-dessus.

Pour l’image 2, la profondeur utilisée est **`depth_2_aligned`** (déjà dans le bundle), masquée par `mask_combined`.


In [4]:
# Nuage de points Image 1 : plan XY normalisé par max(h,w), Z = valeurs brutes de depth_1_masked (même échelle que la depth map)
import open3d as o3d

rgb_img = np.array(image_cropped_1)
if rgb_img.dtype != np.uint8:
    rgb_img = (np.clip(rgb_img, 0, 1) * 255).astype(np.uint8)
h, w = depth_1_masked.shape
if rgb_img.shape[0] != h or rgb_img.shape[1] != w:
    rgb_img = np.asarray(Image.fromarray(rgb_img).resize((w, h), Image.Resampling.LANCZOS), dtype=np.uint8)
xx, yy = np.meshgrid(np.arange(w), np.arange(h), indexing="xy")

zv = depth_1_masked[np.isfinite(depth_1_masked)]
if zv.size == 0:
    raise ValueError("depth_1_masked : aucun pixel valide pour le nuage de points.")
z_min, z_max = float(zv.min()), float(zv.max())
z_span = max(z_max - z_min, 1e-12)
z_std = float(np.std(zv))
n_unique = len(np.unique(np.round(zv.astype(np.float64), 8)))
print(
    f"Img1 — Z brut (depth_map): min={z_min:.6g}, max={z_max:.6g}, span={z_span:.6g}, "
    f"std={z_std:.6g}, valeurs uniques (~8 déc.)={n_unique}"
)
if z_span < 1e-9:
    raise ValueError("Toutes les profondeurs Z sont identiques (span ~ 0) — vérifier depth_1 / masque.")

s_xy = float(max(w, h))
x_n = xx.astype(np.float64) / s_xy
y_n = yy.astype(np.float64) / s_xy
z_n = np.where(
    np.isfinite(depth_1_masked),
    depth_1_masked.astype(np.float64),
    np.nan,
)
points = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3)
tint_1 = np.array([0.92, 0.38, 0.28])
base_rgb = rgb_img.reshape(-1, 3).astype(np.float64) / 255.0
colors = np.clip(0.5 * base_rgb + 0.5 * tint_1, 0.0, 1.0)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)
valid = np.isfinite(points[:, 2])
pcd = pcd.select_by_index(np.where(valid)[0])
z_vis = np.asarray(pcd.points)[:, 2]
print(
    f"Img1 — Z nuage (= depth_map): min={z_vis.min():.6g}, max={z_vis.max():.6g}, "
    f"span={float(z_vis.max() - z_vis.min()):.6g}"
)

cx, cy = 0.5 * (w - 1) / s_xy, 0.5 * (h - 1) / s_xy
axis_size = max(0.08, 0.15 * z_span)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size, origin=[cx, cy, z_min])

# Axe Z (bleu) sur la plage réelle des profondeurs [z_min, z_max]
line_z_pts = np.array([[cx, cy, z_min], [cx, cy, z_max]], dtype=np.float64)
line_z = o3d.geometry.LineSet()
line_z.points = o3d.utility.Vector3dVector(line_z_pts)
line_z.lines = o3d.utility.Vector2iVector(np.array([[0, 1]], dtype=np.int32))
line_z.colors = o3d.utility.Vector3dVector(np.array([[0.15, 0.35, 1.0]], dtype=np.float64))

tick_half = 0.02
tick_z_vals = np.linspace(z_min, z_max, num=5)
tp, tl, tc = [], [], []
k = 0
for zt in tick_z_vals:
    tp.extend([[cx - tick_half, cy, zt], [cx + tick_half, cy, zt]])
    tl.append([k, k + 1])
    tc.append([0.2, 0.45, 1.0])
    k += 2
ticks_z = o3d.geometry.LineSet()
ticks_z.points = o3d.utility.Vector3dVector(np.asarray(tp, dtype=np.float64))
ticks_z.lines = o3d.utility.Vector2iVector(np.asarray(tl, dtype=np.int32))
ticks_z.colors = o3d.utility.Vector3dVector(np.asarray(tc, dtype=np.float64))

o3d.visualization.draw_geometries(
    [pcd, frame, line_z, ticks_z],
    window_name="Image 1 — Nuage (XY norm., Z = depth_1_masked brut)",
)


Img1 — Z brut (depth_map): min=0.375788, max=1, span=0.624212, std=0.123209, valeurs uniques (~8 déc.)=555644
Img1 — Z nuage (= depth_map): min=0.375788, max=1, span=0.624212


In [5]:
# Superposition 3D : XY normalisés, Z = valeurs brutes des depth_map masquées
# Teinte rougeâtre (image 1) vs bleutée (image 2)
import open3d as o3d

h_s, w_s = depth_1_masked.shape
if depth_2_masked.shape != (h_s, w_s):
    raise ValueError("depth_1_masked et depth_2_masked doivent avoir la meme taille.")

s_xy = float(max(h_s, w_s))


def _rgb_cropped_to_depth_grid(rgb_source, h, w):
    arr = np.array(rgb_source)
    if arr.dtype != np.uint8:
        arr = (np.clip(arr, 0, 1) * 255).astype(np.uint8)
    if arr.shape[0] != h or arr.shape[1] != w:
        arr = np.asarray(
            Image.fromarray(arr).resize((w, h), Image.Resampling.LANCZOS),
            dtype=np.uint8,
        )
    return arr


def _pcd_from_masked_depth(z_masked, rgb_uint8, tint_rgb, mix, h_img, w_img, s_xy_, x_offset=0.0):
    """Z = valeurs brutes du tableau de profondeur masqué (même échelle que la depth map)."""
    xx, yy = np.meshgrid(np.arange(w_img), np.arange(h_img), indexing="xy")
    x_n = xx.astype(np.float64) / s_xy_ + float(x_offset)
    y_n = yy.astype(np.float64) / s_xy_
    z_n = np.where(
        np.isfinite(z_masked),
        z_masked.astype(np.float64),
        np.nan,
    )
    pts = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3).astype(np.float64)
    base = rgb_uint8.reshape(-1, 3).astype(np.float64) / 255.0
    tint = np.asarray(tint_rgb, dtype=np.float64).reshape(1, 3)
    col = (1.0 - mix) * base + mix * tint
    col = np.clip(col, 0.0, 1.0)
    valid = np.isfinite(pts[:, 2])
    idx = np.where(valid)[0]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts[idx])
    pcd.colors = o3d.utility.Vector3dVector(col[idx])
    return pcd


rgb1 = _rgb_cropped_to_depth_grid(image_cropped_1, h_s, w_s)
rgb2 = _rgb_cropped_to_depth_grid(image_cropped_2, h_s, w_s)

z1v = depth_1_masked[np.isfinite(depth_1_masked)]
z2v = depth_2_masked[np.isfinite(depth_2_masked)]
if z1v.size == 0 and z2v.size == 0:
    raise ValueError("Aucune profondeur valide pour les nuages.")

# Stats sur le masque (pour axe Z de visualisation et contrôle span)
if z1v.size:
    z1_min, z1_max = float(z1v.min()), float(z1v.max())
else:
    z1_min, z1_max = 0.0, 1.0
if z2v.size:
    z2_min, z2_max = float(z2v.min()), float(z2v.max())
else:
    z2_min, z2_max = 0.0, 1.0
for name, zv in [("depth_1_masked", z1v), ("depth_2_masked", z2v)]:
    if zv.size:
        zu = len(np.unique(np.round(zv.astype(np.float64), 8)))
        print(
            f"{name}: Z brut min={float(zv.min()):.6g}, max={float(zv.max()):.6g}, "
            f"std={float(np.std(zv)):.6g}, uniques≈{zu}"
        )
if z1v.size and max(z1_max - z1_min, 0.0) < 1e-9:
    raise ValueError("depth_1 : Z quasi constante sur le masque.")
if z2v.size and max(z2_max - z2_min, 0.0) < 1e-9:
    raise ValueError("depth_2 : Z quasi constante sur le masque.")

# Même position XY pour les deux nuages (superposition) ; Z = profondeur respective par image.
x_offset_1 = 0.0
x_offset_2 = 0.0

# Teintes (RGB 0-1) : rouge/orange vs cyan/bleu — mélange avec RGB pour différencier les deux images
tint_1 = np.array([0.92, 0.38, 0.28])
tint_2 = np.array([0.25, 0.55, 0.95])

pcd_super_1 = _pcd_from_masked_depth(
    depth_1_masked, rgb1, tint_1, 0.5, h_s, w_s, s_xy, x_offset=x_offset_1
)
pcd_super_2 = _pcd_from_masked_depth(
    depth_2_masked, rgb2, tint_2, 0.5, h_s, w_s, s_xy, x_offset=x_offset_2
)
z_vis_1 = np.asarray(pcd_super_1.points)[:, 2]
z_vis_2 = np.asarray(pcd_super_2.points)[:, 2]
print(
    f"Z (= depth_map) nuage1: [{z_vis_1.min():.6g}, {z_vis_1.max():.6g}] | "
    f"nuage2: [{z_vis_2.min():.6g}, {z_vis_2.max():.6g}]"
)

# Repère + axe Z sur la plage couverte par les deux nuages (profondeurs brutes)
cy = 0.5 * (h_s - 1) / s_xy
cx = 0.5 * (w_s - 1) / s_xy
if z1v.size and z2v.size:
    z_axis_lo = min(z1_min, z2_min)
    z_axis_hi = max(z1_max, z2_max)
elif z1v.size:
    z_axis_lo, z_axis_hi = z1_min, z1_max
else:
    z_axis_lo, z_axis_hi = z2_min, z2_max
z_axis_span = max(z_axis_hi - z_axis_lo, 1e-12)
axis_size = max(0.08, 0.15 * z_axis_span)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=axis_size, origin=[cx, cy, z_axis_lo]
)


def _z_axis_and_ticks(cx_, cy_, z_lo, z_hi):
    lz_pts = np.array([[cx_, cy_, z_lo], [cx_, cy_, z_hi]], dtype=np.float64)
    lz = o3d.geometry.LineSet()
    lz.points = o3d.utility.Vector3dVector(lz_pts)
    lz.lines = o3d.utility.Vector2iVector(np.array([[0, 1]], dtype=np.int32))
    lz.colors = o3d.utility.Vector3dVector(np.array([[0.15, 0.35, 1.0]], dtype=np.float64))

    tick_half = 0.02
    tp, tl, tc = [], [], []
    k = 0
    for zt in np.linspace(z_lo, z_hi, num=5):
        tp.extend([[cx_ - tick_half, cy_, zt], [cx_ + tick_half, cy_, zt]])
        tl.append([k, k + 1])
        tc.append([0.2, 0.45, 1.0])
        k += 2
    tz = o3d.geometry.LineSet()
    tz.points = o3d.utility.Vector3dVector(np.asarray(tp, dtype=np.float64))
    tz.lines = o3d.utility.Vector2iVector(np.asarray(tl, dtype=np.int32))
    tz.colors = o3d.utility.Vector3dVector(np.asarray(tc, dtype=np.float64))
    return lz, tz


line_z, ticks_z = _z_axis_and_ticks(cx, cy, z_axis_lo, z_axis_hi)

win = (
    f"Superposition — Z = depth brutes [{z_axis_lo:.4g}, {z_axis_hi:.4g}] ; "
    f"img1 rouge [{z1_min:.4g}, {z1_max:.4g}] ; img2 bleu [{z2_min:.4g}, {z2_max:.4g}]"
)
o3d.visualization.draw_geometries(
    [pcd_super_1, pcd_super_2, frame, line_z, ticks_z],
    window_name=win,
)

depth_1_masked: Z brut min=0.375788, max=1, std=0.123209, uniques≈555644
depth_2_masked: Z brut min=0.47084, max=1, std=0.120608, uniques≈592204
Z (= depth_map) nuage1: [0.375788, 1] | nuage2: [0.47084, 1]


In [6]:
# Relief 3D Image 1 : XY / max(h,w), Z = depth_1_masked brut
rgb_img = np.asarray(image_cropped_1, dtype=np.uint8)
if rgb_img.ndim == 2:
    rgb_img = np.stack([rgb_img] * 3, axis=-1)
h, w = depth_1_masked.shape
if rgb_img.shape[0] != h or rgb_img.shape[1] != w:
    rgb_img = np.asarray(Image.fromarray(rgb_img).resize((w, h), Image.Resampling.LANCZOS), dtype=np.uint8)

s_xy = float(max(h, w))
cols = np.arange(w, dtype=np.float64)
rows = np.arange(h, dtype=np.float64)
points = np.empty((h, w, 3), dtype=np.float64)
points[:, :, 0] = (cols / s_xy)[np.newaxis, :]
points[:, :, 1] = (rows / s_xy)[:, np.newaxis]
z_geom = np.where(
    np.isfinite(depth_1_masked),
    depth_1_masked.astype(np.float64),
    np.nan,
)
points[:, :, 2] = z_geom
valid = np.isfinite(points[:, :, 2])
points_valid = points[valid]
colors_valid = (rgb_img.reshape(-1, 3) / 255.0).astype(np.float64)[valid.ravel()]
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_valid)
pcd.colors = o3d.utility.Vector3dVector(colors_valid)
o3d.visualization.draw_geometries([pcd], window_name="Image 1 — Relief 3D (Z = depth brut)")


In [7]:
# Nuage de points : Z = depth_1 - depth_2_aligned (différence sur la carte alignée)
h1, w1 = depth_1.shape
depth_2_aligned = np.asarray(depth_2_aligned, dtype=np.float64)
if depth_2_aligned.shape != (h1, w1):
    depth_2_aligned = cv2.resize(
        depth_2_aligned.astype(np.float32), (w1, h1), interpolation=cv2.INTER_LINEAR
    ).astype(np.float64)
diff_z = depth_1.astype(np.float64) - depth_2_aligned
# On garde uniquement les pixels objet (mask_obj) pour éviter le bruit du fond
diff_masked = np.where(mask_combined == 255, diff_z, np.nan)

rgb_diff = np.array(image_cropped_1)
if rgb_diff.dtype != np.uint8:
    rgb_diff = (np.clip(rgb_diff, 0, 1) * 255).astype(np.uint8)
if rgb_diff.shape[0] != h1 or rgb_diff.shape[1] != w1:
    rgb_diff = np.asarray(Image.fromarray(rgb_diff).resize((w1, h1), Image.Resampling.LANCZOS), dtype=np.uint8)

xx_d, yy_d = np.meshgrid(np.arange(w1), np.arange(h1))
pts_diff = np.stack([xx_d, yy_d, diff_masked], axis=-1).reshape(-1, 3)
colors_diff = rgb_diff.reshape(-1, 3) / 255.0
pcd_diff = o3d.geometry.PointCloud()
pcd_diff.points = o3d.utility.Vector3dVector(pts_diff)
pcd_diff.colors = o3d.utility.Vector3dVector(colors_diff)
valid_d = np.isfinite(pts_diff[:, 2])
pcd_diff = pcd_diff.select_by_index(np.where(valid_d)[0])

# Option : coordonnées spatiales pour un relief lisible (Z = différence, valeurs brutes)
scale_xy_d = 1.0 / w1
pts_sp = np.column_stack([
    pts_diff[valid_d, 0] * scale_xy_d,
    pts_diff[valid_d, 1] * scale_xy_d,
    pts_diff[valid_d, 2],
])
pcd_diff.points = o3d.utility.Vector3dVector(pts_sp)

o3d.visualization.draw_geometries(
    [pcd_diff],
    window_name="Nuage de points : Z = Image 1 − Image 2 (depth_2_aligned)"
)

### Géométrisation — Différence (Z = Image 1 − Image 2)

Le nuage `pcd_diff` utilise **depth_1 − depth_2_aligned** (carte 2 alignée sur la grille de l’image 1). Même pipeline que pour l’image 1 : downsampling, enrichissement par points en Z=0, puis reconstruction de surface (mesh + normales).

In [8]:
# Downsample du nuage différence (pcd_diff) pour alléger la géométrisation
voxel_size_diff = 0.002
pcd_diff_ds = pcd_diff.voxel_down_sample(voxel_size_diff)
o3d.visualization.draw_geometries(
    [pcd_diff_ds],
    window_name="Différence (Image 1 − Image 2) — Relief 3D (downsampled)"
)

In [9]:
# Reconstruction de surface par Poisson sur le nuage différence (comme Image 1)

# (On n'utilise plus pyvista ici, mais le Poisson surface reconstruction d'Open3D)
# Attention : nécessite des normales cohérentes sur le nuage

# Nettoyage et recalcul des normales
pcd_diff_pv = pcd_diff_ds.voxel_down_sample(0.005)
pcd_diff_pv.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30),
    fast_normal_computation=True,
    
)
# Oriente les normales de façon cohérente
pcd_diff_pv.orient_normals_consistent_tangent_plane(k=10)
pcd_diff_pv.orient_normals_to_align_with_direction()
# Reconstruction par Poisson
mesh_diff, densities_diff = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_diff_pv, depth=8
)

# Optionnel : suppression des triangles de faible densité (pour nettoyer)
densities_diff = np.asarray(densities_diff)
density_threshold = np.quantile(densities_diff, 0.02)
vertices_to_remove = densities_diff < density_threshold
mesh_diff.remove_vertices_by_mask(vertices_to_remove)
mesh_diff.compute_vertex_normals()
normals_diff = np.asarray(mesh_diff.vertex_normals)
verts_diff = np.asarray(mesh_diff.vertices)

# Lignes des normales (indices corrects)
step_n = max(len(normals_diff) // 300, 1)
line_pts_diff = []
for i in range(0, len(normals_diff), step_n):
    start = verts_diff[i]
    end = start + normals_diff[i] * -0.02
    line_pts_diff.append(start)
    line_pts_diff.append(end)
n_ln = len(line_pts_diff) // 2
line_idx_diff = np.array([[2*k, 2*k+1] for k in range(n_ln)], dtype=np.int32)
line_set_diff = o3d.geometry.LineSet()
line_set_diff.points = o3d.utility.Vector3dVector(np.asarray(line_pts_diff))
line_set_diff.lines = o3d.utility.Vector2iVector(line_idx_diff)
line_set_diff.colors = o3d.utility.Vector3dVector(np.tile([1, 0, 0], (n_ln, 1)))

# Arêtes de bord : arêtes qui n'appartiennent qu'à un seul triangle
tris_diff = np.asarray(mesh_diff.triangles)
edge_count = {}
for (a, b, c) in tris_diff:
    for u, v in [(a, b), (b, c), (c, a)]:
        e = (min(u, v), max(u, v))
        edge_count[e] = edge_count.get(e, 0) + 1
boundary_edges = np.array([e for e, c in edge_count.items() if c == 1], dtype=np.int32)
line_set_boundary = o3d.geometry.LineSet()
line_set_boundary.points = mesh_diff.vertices
line_set_boundary.lines = o3d.utility.Vector2iVector(boundary_edges)
line_set_boundary.paint_uniform_color([0, 1, 0])  # vert = bord du mesh

geoms_diff = [mesh_diff]
if n_ln > 0:
    geoms_diff.append(line_set_diff)
geoms_diff.append(line_set_boundary)
o3d.visualization.draw_geometries(
    geoms_diff,
    window_name="Différence (Image 1 − Image 2) — Mesh Poisson + normales + bord"
)

### Visualisations 3D — Image 2 (25 g)

Nuage de points et relief 3D pour la **deuxième image** : Z = `depth_2_masked`, construit à partir de **`depth_2_aligned`** dans le masque commun (`mask_combined`), couleurs depuis `image_cropped_2`. Permet de comparer les deux prises (33 g vs 25 g) sur la même échelle de profondeur que l’image 1.

In [10]:
# Nuage de points Image 2 : XY normalisés, Z = depth_2_masked brut (même échelle que la depth map)
# + teinte bleue pour cohérence avec la superposition deux images
rgb_img_2 = np.array(image_cropped_2)
if rgb_img_2.dtype != np.uint8:
    rgb_img_2 = (np.clip(rgb_img_2, 0, 1) * 255).astype(np.uint8)
h2, w2 = depth_2_masked.shape
if rgb_img_2.shape[0] != h2 or rgb_img_2.shape[1] != w2:
    rgb_img_2 = np.asarray(
        Image.fromarray(rgb_img_2).resize((w2, h2), Image.Resampling.LANCZOS),
        dtype=np.uint8,
    )

xx_2, yy_2 = np.meshgrid(np.arange(w2), np.arange(h2), indexing="xy")
s_xy_2 = float(max(w2, h2))
x_n = xx_2.astype(np.float64) / s_xy_2
y_n = yy_2.astype(np.float64) / s_xy_2

zv2 = depth_2_masked[np.isfinite(depth_2_masked)]
if zv2.size == 0:
    raise ValueError("depth_2_masked : aucun pixel valide.")
z2_lo, z2_hi = float(zv2.min()), float(zv2.max())
z_span2 = max(z2_hi - z2_lo, 1e-12)
z_n = np.where(
    np.isfinite(depth_2_masked),
    depth_2_masked.astype(np.float64),
    np.nan,
)
points_2 = np.stack([x_n, y_n, z_n], axis=-1).reshape(-1, 3)

tint_2 = np.array([0.25, 0.55, 0.95])
base_2 = rgb_img_2.reshape(-1, 3).astype(np.float64) / 255.0
colors_2 = np.clip(0.5 * base_2 + 0.5 * tint_2, 0.0, 1.0)

pcd_2 = o3d.geometry.PointCloud()
pcd_2.points = o3d.utility.Vector3dVector(points_2)
pcd_2.colors = o3d.utility.Vector3dVector(colors_2)
valid_2 = np.isfinite(points_2[:, 2])
pcd_2 = pcd_2.select_by_index(np.where(valid_2)[0])

cx2, cy2 = 0.5 * (w2 - 1) / s_xy_2, 0.5 * (h2 - 1) / s_xy_2
axis_size = max(0.08, 0.15 * z_span2)
frame_2 = o3d.geometry.TriangleMesh.create_coordinate_frame(size=axis_size, origin=[cx2, cy2, z2_lo])
line_z_pts = np.array([[cx2, cy2, z2_lo], [cx2, cy2, z2_hi]], dtype=np.float64)
line_z_2 = o3d.geometry.LineSet()
line_z_2.points = o3d.utility.Vector3dVector(line_z_pts)
line_z_2.lines = o3d.utility.Vector2iVector(np.array([[0, 1]], dtype=np.int32))
line_z_2.colors = o3d.utility.Vector3dVector(np.array([[0.15, 0.35, 1.0]], dtype=np.float64))
tick_half = 0.02
tp, tl, tc = [], [], []
k = 0
for zt in np.linspace(z2_lo, z2_hi, num=5):
    tp.extend([[cx2 - tick_half, cy2, zt], [cx2 + tick_half, cy2, zt]])
    tl.append([k, k + 1])
    tc.append([0.2, 0.45, 1.0])
    k += 2
ticks_z_2 = o3d.geometry.LineSet()
ticks_z_2.points = o3d.utility.Vector3dVector(np.asarray(tp, dtype=np.float64))
ticks_z_2.lines = o3d.utility.Vector2iVector(np.asarray(tl, dtype=np.int32))
ticks_z_2.colors = o3d.utility.Vector3dVector(np.asarray(tc, dtype=np.float64))

o3d.visualization.draw_geometries(
    [pcd_2, frame_2, line_z_2, ticks_z_2],
    window_name="Image 2 — Nuage (XY norm., Z = depth_2 brut, teinte bleue)",
)

In [11]:
# Géométrisation Poisson — Image 2 (carte depth_2_masked)
# Reconstruction de surface par Poisson sur le nuage de la deuxième carte de profondeur

# Downsample du nuage Image 2 pour alléger la reconstruction
voxel_size_2 = 0.002
pcd_2_ds = pcd_2_spatial.voxel_down_sample(voxel_size_2)
pcd_2_poisson = pcd_2_ds.voxel_down_sample(0.005)

# Estimation et orientation des normales (requis pour Poisson)
pcd_2_poisson.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.02, max_nn=30),
    fast_normal_computation=True,
)
pcd_2_poisson.orient_normals_consistent_tangent_plane(k=10)
#pcd_2_poisson.orient_normals_to_align_with_direction()


# Reconstruction de surface par Poisson
mesh_2, densities_2 = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_2_poisson, depth=8
)

# Suppression des triangles de faible densité (nettoyage)
densities_2 = np.asarray(densities_2)
density_threshold_2 = np.quantile(densities_2, 0.02)
vertices_to_remove_2 = densities_2 < density_threshold_2
mesh_2.remove_vertices_by_mask(vertices_to_remove_2)
mesh_2.compute_vertex_normals()


# Visualisation : mesh Poisson + arêtes de bord
tris_2 = np.asarray(mesh_2.triangles)
edge_count_2 = {}
for (a, b, c) in tris_2:
    for u, v in [(a, b), (b, c), (c, a)]:
        e = (min(u, v), max(u, v))
        edge_count_2[e] = edge_count_2.get(e, 0) + 1
boundary_edges_2 = np.array([e for e, c in edge_count_2.items() if c == 1], dtype=np.int32)
line_set_boundary_2 = o3d.geometry.LineSet()
line_set_boundary_2.points = mesh_2.vertices
line_set_boundary_2.lines = o3d.utility.Vector2iVector(boundary_edges_2)
line_set_boundary_2.paint_uniform_color([0, 1, 0])

o3d.visualization.draw_geometries(
    [mesh_2, line_set_boundary_2],
    window_name="Image 2 — Mesh Poisson (depth_2_masked, géométrie 3D)",
)

NameError: name 'pcd_2_spatial' is not defined

In [ ]:
# Superposition du mesh (Image 1) par Poisson
# Version robuste: reconstruction allégée pour éviter les crash kernel
# + contrôle manifold / watertight + volume (mesh ou fallback voxel)


def _prepare_pcd_for_poisson(pcd, voxel_size=0.005, max_points=70000):
    """Downsample + sous-échantillonnage borné pour limiter RAM/CPU."""
    p = pcd.voxel_down_sample(voxel_size)
    pts = np.asarray(p.points)
    if pts.shape[0] > max_points:
        idx = np.linspace(0, pts.shape[0] - 1, max_points, dtype=np.int64)
        p = p.select_by_index(idx.tolist())
    return p


def _poisson_mesh_from_pcd_safe(pcd, voxel_size=0.005, depth=6, density_quantile=0.03):
    """Reconstruction Poisson plus stable (moins agressive)."""
    p = _prepare_pcd_for_poisson(pcd, voxel_size=voxel_size)
    p.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=30),
        fast_normal_computation=True,
    )
    try:
        p.orient_normals_consistent_tangent_plane(k=10)
    except Exception:
        # Nuages quasi-planaires (ex: pcd_0) peuvent échouer avec QHull.
        p.orient_normals_to_align_with_direction([0.0, 0.0, -1.0])

    # n_threads=1 limite les pics mémoire/CPU sur certaines machines
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        p, depth=depth, n_threads=1
    )

    dens = np.asarray(densities)
    if density_quantile is not None and density_quantile > 0.0:
        thr = np.quantile(dens, density_quantile)
        mesh.remove_vertices_by_mask(dens < thr)

    # Nettoyage topologique sans API tensor (source fréquente de crash natif)
    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.remove_degenerate_triangles()
    mesh.remove_non_manifold_edges()
    mesh.remove_unreferenced_vertices()
    mesh.compute_vertex_normals()
    return mesh, p


def _boundary_lines(mesh, max_lines=20000):
    """LineSet des arêtes de bord; tronqué pour visualisation fluide."""
    tris = np.asarray(mesh.triangles)
    edge_count = {}
    for (a, b, c) in tris:
        for u, v in [(a, b), (b, c), (c, a)]:
            e = (min(u, v), max(u, v))
            edge_count[e] = edge_count.get(e, 0) + 1

    boundary_edges = np.array([e for e, c in edge_count.items() if c == 1], dtype=np.int32)
    n_total = int(boundary_edges.shape[0])
    if n_total > max_lines:
        keep = np.linspace(0, n_total - 1, max_lines, dtype=np.int64)
        boundary_edges_vis = boundary_edges[keep]
    else:
        boundary_edges_vis = boundary_edges

    ls = o3d.geometry.LineSet()
    ls.points = mesh.vertices
    if boundary_edges_vis.size > 0:
        ls.lines = o3d.utility.Vector2iVector(boundary_edges_vis)
    else:
        ls.lines = o3d.utility.Vector2iVector(np.empty((0, 2), dtype=np.int32))
    ls.paint_uniform_color([0.0, 1.0, 0.0])
    return ls, n_total


def _voxel_volume_from_pcd(pcd, voxel_size=0.01):
    vg = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size=voxel_size)
    return len(vg.get_voxels()) * (voxel_size ** 3)


def _count_boundary_edges(mesh):
    tris = np.asarray(mesh.triangles)
    edge_count = {}
    for (a, b, c) in tris:
        for u, v in [(a, b), (b, c), (c, a)]:
            e = (min(u, v), max(u, v))
            edge_count[e] = edge_count.get(e, 0) + 1
    return int(sum(1 for c in edge_count.values() if c == 1))


def _seal_mesh_holes_safe(mesh, hole_size=1e6, enable_tensor=False):
    """Tentative robuste de fermeture des trous (tensor fill_holes), avec fallback sûr."""
    n_before = _count_boundary_edges(mesh)
    if n_before == 0:
        return mesh, n_before, n_before, True

    # Désactivé par défaut pour éviter les crash kernel sur gros meshes.
    if not enable_tensor:
        return mesh, n_before, n_before, False

    ok = False
    mesh_out = mesh
    try:
        tmesh = o3d.t.geometry.TriangleMesh.from_legacy(mesh)
        tmesh = tmesh.fill_holes(hole_size=float(hole_size))
        mesh_out = tmesh.to_legacy()
        mesh_out.remove_duplicated_vertices()
        mesh_out.remove_duplicated_triangles()
        mesh_out.remove_degenerate_triangles()
        mesh_out.remove_non_manifold_edges()
        mesh_out.remove_unreferenced_vertices()
        mesh_out.compute_triangle_normals()
        mesh_out.compute_vertex_normals()
        ok = True
    except Exception:
        # Fallback: on garde le mesh original si fill_holes n'est pas dispo/échoue.
        mesh_out = mesh

    n_after = _count_boundary_edges(mesh_out)
    return mesh_out, n_before, n_after, ok


def _remove_aberrant_triangles(mesh, edge_quantile=0.995, z_sigma=4.0):
    """Supprime les triangles aberrants (arêtes très longues / pics en Z)."""
    tris = np.asarray(mesh.triangles)
    if tris.size == 0:
        return mesh, 0, 0.0

    verts = np.asarray(mesh.vertices)
    tri_pts = verts[tris]
    e01 = np.linalg.norm(tri_pts[:, 0, :] - tri_pts[:, 1, :], axis=1)
    e12 = np.linalg.norm(tri_pts[:, 1, :] - tri_pts[:, 2, :], axis=1)
    e20 = np.linalg.norm(tri_pts[:, 2, :] - tri_pts[:, 0, :], axis=1)
    emax = np.maximum(np.maximum(e01, e12), e20)
    thr_edge = float(np.quantile(emax, edge_quantile))

    cz = np.mean(tri_pts[:, :, 2], axis=1)
    med = float(np.median(cz))
    mad = float(np.median(np.abs(cz - med))) + 1e-12
    robust_sigma = 1.4826 * mad

    bad = (emax > thr_edge) | (np.abs(cz - med) > (z_sigma * robust_sigma))
    n_bad = int(np.count_nonzero(bad))
    if n_bad > 0:
        mesh.remove_triangles_by_mask(bad)
        mesh.remove_unreferenced_vertices()
        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
        mesh.remove_duplicated_vertices()

    return mesh, n_bad, thr_edge


def _patch_small_holes_from_boundary_loops(mesh, max_loop_edges=18, max_diameter_ratio=0.05):
    """Colmate localement les petits trous: boucles de bord internes -> patch triangulaire."""
    tris = np.asarray(mesh.triangles)
    if tris.size == 0:
        return mesh, 0, 0, 0

    # Arêtes de bord (occurrence unique)
    edge_count = {}
    for (a, b, c) in tris:
        for u, v in [(a, b), (b, c), (c, a)]:
            e = (min(u, v), max(u, v))
            edge_count[e] = edge_count.get(e, 0) + 1
    boundary_edges = [e for e, c in edge_count.items() if c == 1]
    if not boundary_edges:
        return mesh, 0, 0, 0

    # Graphe des arêtes de bord
    adj = {}
    for u, v in boundary_edges:
        adj.setdefault(u, []).append(v)
        adj.setdefault(v, []).append(u)

    # Extraction de boucles
    visited = set()
    loops = []
    for e in boundary_edges:
        if e in visited:
            continue
        u0, v0 = e
        loop = [u0, v0]
        visited.add(e)
        prev, cur = u0, v0
        guard = 0
        while guard < 10000:
            guard += 1
            nbrs = adj.get(cur, [])
            nxts = [n for n in nbrs if n != prev]
            if not nxts:
                break
            nxt = nxts[0]
            e2 = (min(cur, nxt), max(cur, nxt))
            if e2 in visited and nxt != loop[0]:
                break
            loop.append(nxt)
            visited.add(e2)
            prev, cur = cur, nxt
            if cur == loop[0]:
                loops.append(loop[:-1])
                break

    if not loops:
        return mesh, 0, 0, 0

    verts = np.asarray(mesh.vertices)
    bb = mesh.get_axis_aligned_bounding_box()
    diag = float(np.linalg.norm(bb.get_extent()))
    max_diam = max(1e-6, max_diameter_ratio * diag)

    tri_add = []
    n_filled = 0

    for loop in loops:
        uniq = []
        seen = set()
        for vid in loop:
            if vid not in seen:
                uniq.append(int(vid))
                seen.add(int(vid))
        if len(uniq) < 3 or len(uniq) > max_loop_edges:
            continue

        pts = verts[np.asarray(uniq, dtype=np.int64)]
        if pts.shape[0] < 3:
            continue
        # Ignore les grandes ouvertures (bord externe)
        diam = float(np.linalg.norm(pts.max(axis=0) - pts.min(axis=0)))
        if diam > max_diam:
            continue

        c = pts.mean(axis=0)
        X = pts - c
        cov = X.T @ X
        w, V = np.linalg.eigh(cov)
        order = np.argsort(w)
        e1 = V[:, order[2]]
        e2 = V[:, order[1]]
        uv = np.column_stack([X @ e1, X @ e2])
        ang = np.arctan2(uv[:, 1], uv[:, 0])
        ord_idx = np.argsort(ang)
        loop_ord = [uniq[i] for i in ord_idx.tolist()]

        # Triangulation en éventail pour petits trous
        v0 = loop_ord[0]
        for i in range(1, len(loop_ord) - 1):
            tri_add.append([v0, loop_ord[i], loop_ord[i + 1]])
        n_filled += 1

    n_before = _count_boundary_edges(mesh)
    if tri_add:
        all_tris = np.vstack([np.asarray(mesh.triangles), np.asarray(tri_add, dtype=np.int32)])
        mesh.triangles = o3d.utility.Vector3iVector(all_tris)
        mesh.remove_degenerate_triangles()
        mesh.remove_duplicated_triangles()
        mesh.remove_unreferenced_vertices()
        mesh.compute_triangle_normals()
        mesh.compute_vertex_normals()
    n_after = _count_boundary_edges(mesh)

    return mesh, n_filled, n_before, n_after


def _triangulate_planar_pcd(pcd, voxel_size=0.02):
    """Triangulation d'un nuage quasi-plan avec maillage plus dense."""
    p = pcd.voxel_down_sample(max(0.25 * voxel_size, 1e-4))
    p.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=max(2.5 * voxel_size, 0.01), max_nn=80),
        fast_normal_computation=True,
    )
    # On inverse explicitement vers -Z pour la face du plan.
    p.orient_normals_to_align_with_direction([0.0, 0.0, -1.0])

    # Multi-rayons pour limiter les trous de triangulation.
    radii = o3d.utility.DoubleVector([
        max(0.7 * voxel_size, 8e-4),
        max(1.1 * voxel_size, 1.2e-3),
        max(1.6 * voxel_size, 1.6e-3),
        max(2.2 * voxel_size, 2.2e-3),
    ])
    mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(p, radii)

    # Fallback alpha-shape si le maillage est trop parcellaire.
    if len(mesh.triangles) < max(100, int(0.6 * len(p.points))):
        alpha = max(2.5 * voxel_size, 0.01)
        mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(p, alpha)

    mesh.remove_duplicated_vertices()
    mesh.remove_duplicated_triangles()
    mesh.remove_degenerate_triangles()
    mesh.remove_unreferenced_vertices()

    tri_labels, tri_counts, _ = mesh.cluster_connected_triangles()
    tri_labels = np.asarray(tri_labels)
    tri_counts = np.asarray(tri_counts)
    if tri_counts.size > 0:
        keep_label = int(np.argmax(tri_counts))
        mesh.remove_triangles_by_mask(tri_labels != keep_label)
        mesh.remove_unreferenced_vertices()

    mesh.compute_triangle_normals()
    mesh.compute_vertex_normals()
    vnorm = np.asarray(mesh.vertex_normals)
    mesh.vertex_normals = o3d.utility.Vector3dVector(-vnorm)
    tnorm = np.asarray(mesh.triangle_normals)
    mesh.triangle_normals = o3d.utility.Vector3dVector(-tnorm)
    return mesh


# Pipeline déplacé en cellules séparées pour améliorer la lisibilité.
# Exécuter les cellules suivantes dans l'ordre.


In [ ]:
# Cellule 1/3 - Construction pcd_0, alignement à line_b1, reconstruction Poisson
# Mode safe pour éviter les crash kernel (RAM/CPU)
poisson_depth_ref = 6
poisson_depth_main = 7
safe_density_q = 0.03
voxel_pcd0 = 0.001
pts0 = np.asarray(pcd_super_1.points, dtype=np.float64).copy()
pts0[:, 2] = 0.0
pcd_0 = o3d.geometry.PointCloud()
pcd_0.points = o3d.utility.Vector3dVector(pts0)
pcd_0 = pcd_0.voxel_down_sample(voxel_size=voxel_pcd0)

# Référence line_b1 sur Image 1 seule (sans pcd_0)
mesh_ref_1, _ = _poisson_mesh_from_pcd_safe(
    pcd_super_1, voxel_size=0.005, depth=poisson_depth_ref, density_quantile=safe_density_q
)
line_b1_ref, _ = _boundary_lines(mesh_ref_1, max_lines=20000)

b_ref = np.asarray(line_b1_ref.points)
bxy_ref = b_ref[:, :2]
bz_ref = b_ref[:, 2]

p_all = np.asarray(pcd_0.points)
p_xy = p_all[:, :2]
z_align = 0.0
if p_xy.shape[0] > 0 and bxy_ref.shape[0] > 0:
    ref_tree = o3d.geometry.PointCloud()
    ref_tree.points = o3d.utility.Vector3dVector(
        np.hstack([bxy_ref, np.zeros((bxy_ref.shape[0], 1), dtype=np.float64)])
    )
    kdt_ref = o3d.geometry.KDTreeFlann(ref_tree)

    # Champ Z lisse : k plus proches sur la bordure (XY) + poids gaussiens (courbes arrondies,
    # au lieu d'un plan constant = médiane des plus proches voisins).
    extent_xy = float(np.ptp(p_xy, axis=0).max()) if p_xy.shape[0] else 1.0
    k_bd = int(min(32, max(4, bxy_ref.shape[0])))
    sigma_xy = max(2.5 * voxel_pcd0, 0.04 * extent_xy)

    z_smooth = np.zeros(p_xy.shape[0], dtype=np.float64)
    for i in range(p_xy.shape[0]):
        q = np.array([p_xy[i, 0], p_xy[i, 1], 0.0], dtype=np.float64)
        kk, idx, d2_nn = kdt_ref.search_knn_vector_3d(q, k_bd)
        if kk > 0:
            d2 = np.asarray(d2_nn[:kk], dtype=np.float64)
            d = np.sqrt(np.maximum(d2, 1e-18))
            w = np.exp(-(d ** 2) / (2.0 * sigma_xy ** 2))
            jj = np.asarray(idx[:kk], dtype=np.int64)
            z_smooth[i] = float(np.dot(w, bz_ref[jj]) / np.sum(w))

    z_align = float(np.median(z_smooth))
    p_all[:, 2] = z_smooth
    pcd_0.points = o3d.utility.Vector3dVector(p_all)
    print(
        f"[Alignement pcd_0] Z lisse (Gauss kNN k={k_bd}, sigma_xy={sigma_xy:.5g}) | "
        f"médiane={z_align:.6g} | écart-type Z={float(np.std(z_smooth)):.5g}"
    )
else:
    if p_xy.shape[0] > 0:
        print(
            f"[Alignement pcd_0] line_b1_ref indisponible ou vide — Z inchangé (z_align={z_align:.6g})"
        )

pcd_0.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=max(2.5 * voxel_pcd0, 0.01), max_nn=80),
    fast_normal_computation=True,
)
pcd_0.orient_normals_to_align_with_direction([0.0, 0.0, -1.0])

# Poisson séparé: corps (pcd_super_1) + cap (pcd_0) pour favoriser une fermeture watertight
mesh_body_1, pcd_super_1_ds = _poisson_mesh_from_pcd_safe(
    pcd_super_1, voxel_size=0.005, depth=poisson_depth_main, density_quantile=safe_density_q
)
mesh_cap_0, pcd_0_ds = _poisson_mesh_from_pcd_safe(
    pcd_0, voxel_size=0.01, depth=6, density_quantile=0.0
)

mesh_super_1 = mesh_body_1 + mesh_cap_0
mesh_super_1.remove_duplicated_vertices()
mesh_super_1.remove_duplicated_triangles()
mesh_super_1.remove_degenerate_triangles()
mesh_super_1.remove_non_manifold_edges()
mesh_super_1.remove_unreferenced_vertices()
mesh_super_1.paint_uniform_color([0.92, 0.38, 0.28])
mesh_super_1 = _flip_mesh_normals(mesh_super_1)
print(
    f"[Poisson split] body_tri={len(mesh_body_1.triangles)} | "
    f"cap_tri={len(mesh_cap_0.triangles)} | merged_tri={len(mesh_super_1.triangles)}"
)

tri_labels, tri_counts, _ = mesh_super_1.cluster_connected_triangles()
tri_labels = np.asarray(tri_labels)
tri_counts = np.asarray(tri_counts)
if tri_counts.size > 0:
    keep_label = int(np.argmax(tri_counts))
    mesh_super_1.remove_triangles_by_mask(tri_labels != keep_label)
    mesh_super_1.remove_unreferenced_vertices()

# fill_holes tensor est coûteux: on ne l'active que si le mesh reste modéré
tri_count = len(mesh_super_1.triangles)
boundary_pre = _count_boundary_edges(mesh_super_1)
if tri_count <= 250000 and boundary_pre <= 4000:
    mesh_super_1, n_hole_before, n_hole_after, hole_close_ok = _seal_mesh_holes_safe(
        mesh_super_1, hole_size=2e5, enable_tensor=True
    )
else:
    n_hole_before, n_hole_after, hole_close_ok = boundary_pre, boundary_pre, False
    print(
        f"[Hole sealing] SKIP (triangles={tri_count}, boundary={boundary_pre}) pour stabilité kernel"
    )

mesh_super_1, n_bad_tri, thr_edge = _remove_aberrant_triangles(
    mesh_super_1, edge_quantile=0.999, z_sigma=4.5
)

# Colmatage local des petits trous internes (souvent visibles en lignes vertes isolées)
mesh_super_1, n_loops_filled, n_b_before_patch, n_b_after_patch = _patch_small_holes_from_boundary_loops(
    mesh_super_1, max_loop_edges=18, max_diameter_ratio=0.05
)

print(f"[Hole sealing] ok={hole_close_ok} | boundary_edges: {n_hole_before} -> {n_hole_after}")
print(f"[Outlier triangles] supprimés={n_bad_tri} | seuil_arête={thr_edge:.4g}")
print(
    f"[Small-hole patch] loops_filled={n_loops_filled} | "
    f"boundary_edges: {n_b_before_patch} -> {n_b_after_patch}"
)

mesh_super_1.compute_triangle_normals()
mesh_super_1.compute_vertex_normals()

# Inversion explicite des normales du modèle rouge
# vnorm_red = np.asarray(mesh_super_1.vertex_normals)
# mesh_super_1.vertex_normals = o3d.utility.Vector3dVector(-vnorm_red)
# tnorm_red = np.asarray(mesh_super_1.triangle_normals)
# mesh_super_1.triangle_normals = o3d.utility.Vector3dVector(-tnorm_red)

line_b1, n_b1 = _boundary_lines(mesh_super_1, max_lines=20000)

[Alignement pcd_0] Z lisse (Gauss kNN k=32, sigma_xy=0.027278) | médiane=0.620892 | écart-type Z=0.12226
[Poisson split] body_tri=57361 | cap_tri=15262 | merged_tri=72623
[Open3D WARNING] Ignoring attribute 'normals' for TensorMap with primary key 'indices'
[Hole sealing] ok=True | boundary_edges: 283 -> 46
[Outlier triangles] supprimés=58 | seuil_arête=0.1542
[Small-hole patch] loops_filled=8 | boundary_edges: 104 -> 78


In [ ]:
# Cellule 2/3 - Diagnostics, volume, ancrage des points rouges et interpolation des gris
pts_ds = np.asarray(pcd_super_1_ds.points)
near_plan = int(np.count_nonzero(np.abs(pts_ds[:, 2] - z_align) <= (0.6 * voxel_pcd0)))
print(
    f"Image 1 — edge_manifold={mesh_super_1.is_edge_manifold()} | "
    f"vertex_manifold={mesh_super_1.is_vertex_manifold()} | "
    f"watertight={mesh_super_1.is_watertight()} | boundary_edges={n_b1}"
)
print(f"[Contrôle pcd_0] points proches du plan dans l'entrée Poisson: {near_plan}/{len(pts_ds)}")

if mesh_super_1.is_watertight():
    try:
        print(f"Image 1 — volume mesh = {mesh_super_1.get_volume():.6g}")
    except Exception as e:
        print(f"Image 1 — volume mesh indisponible ({e})")
else:
    v_vox = _voxel_volume_from_pcd(pcd_super_1_ds, voxel_size=0.01)
    print(f"Image 1 — non watertight, volume voxel fallback = {v_vox:.6g}")

lines_arr = np.asarray(line_b1.lines)
if lines_arr.size:
    v_on_edge = np.unique(lines_arr.ravel())
    bxyz = np.asarray(line_b1.points)[v_on_edge]
    bxy = bxyz[:, :2]
    bz = bxyz[:, 2]
else:
    bxy = np.empty((0, 2), dtype=np.float64)
    bz = np.empty((0,), dtype=np.float64)

p_all = np.asarray(pcd_0.points)
p_xy = p_all[:, :2]
n_p = int(p_xy.shape[0])
hit = np.zeros(n_p, dtype=bool)
extent_xy = float(np.ptp(p_xy, axis=0).max()) if n_p else 1.0
tol_xy = max(1e-9, 5.5 * voxel_pcd0, 1e-4 * extent_xy)

if n_p and bxy.shape[0] > 0:
    pcd_xy_tree = o3d.geometry.PointCloud()
    pcd_xy_tree.points = o3d.utility.Vector3dVector(
        np.hstack([p_xy, np.zeros((n_p, 1), dtype=np.float64)])
    )
    kdt = o3d.geometry.KDTreeFlann(pcd_xy_tree)
    for j in range(bxy.shape[0]):
        q = np.array([bxy[j, 0], bxy[j, 1], 0.0], dtype=np.float64)
        kk, idx, _ = kdt.search_radius_vector_3d(q, tol_xy)
        if kk > 0:
            hit[np.asarray(idx, dtype=np.int64)] = True

# Z : champ Gauss kNN sur *toute* line_b1 (mesh fusionné), pour *tous* les points.
# L'ancienne IDW (gris depuis l'anneau rouge seulement) effaçait le lissage de la cellule 1.
z_prior = p_all[:, 2].copy()
z_vals = z_prior.copy()
blend_cell1_z = 0.25  # part du Z cellule 1 (line_b1_ref) conservée ; 0 = 100 % line_b1 mesh
if n_p > 0 and bxy.shape[0] > 0:
    b_tree = o3d.geometry.PointCloud()
    b_tree.points = o3d.utility.Vector3dVector(
        np.hstack([bxy, np.zeros((bxy.shape[0], 1), dtype=np.float64)])
    )
    b_kdt = o3d.geometry.KDTreeFlann(b_tree)
    k_bd2 = int(min(32, max(4, bxy.shape[0])))
    sigma_xy_b = max(2.5 * voxel_pcd0, 0.04 * extent_xy)
    z_b = np.zeros(n_p, dtype=np.float64)
    for i in range(n_p):
        q = np.array([p_xy[i, 0], p_xy[i, 1], 0.0], dtype=np.float64)
        kk, idx_nn, d2_nn = b_kdt.search_knn_vector_3d(q, k_bd2)
        if kk > 0:
            d2 = np.asarray(d2_nn[:kk], dtype=np.float64)
            d = np.sqrt(np.maximum(d2, 1e-18))
            w = np.exp(-(d ** 2) / (2.0 * sigma_xy_b ** 2))
            jj = np.asarray(idx_nn[:kk], dtype=np.int64)
            z_b[i] = float(np.dot(w, bz[jj]) / np.sum(w))
    z_vals = (1.0 - blend_cell1_z) * z_b + blend_cell1_z * z_prior
    print(
        f"[pcd_0 Z] Gauss kNN sur line_b1 mesh: k={k_bd2}, sigma_xy={sigma_xy_b:.5g}, "
        f"mélange cell.1={blend_cell1_z:.2f} | ΔZ médiane vs prior={float(np.median(np.abs(z_vals - z_prior))):.5g}"
    )

p_all[:, 2] = z_vals
pcd_0.points = o3d.utility.Vector3dVector(p_all)
base_rgb = np.tile(np.array([0.5, 0.5, 0.5], dtype=np.float64), (n_p, 1))
base_rgb[hit] = [1.0, 0.0, 0.0]
pcd_0.colors = o3d.utility.Vector3dVector(base_rgb)
print(
    f"pcd_0: {n_p} pts — rouges={int(hit.sum())}, gris={int((~hit).sum())} | "
    f"rouge = proche bord (affichage), Z = champ lisse sur line_b1 + prior cell.1"
)

Image 1 — edge_manifold=False | vertex_manifold=False | watertight=False | boundary_edges=78
[Contrôle pcd_0] points proches du plan dans l'entrée Poisson: 193/35225
Image 1 — non watertight, volume voxel fallback = 0.007655
[pcd_0 Z] Gauss kNN sur line_b1 mesh: k=32, sigma_xy=0.027278, mélange cell.1=0.25 | ΔZ médiane vs prior=0.073166
pcd_0: 296900 pts — rouges=0, gris=296900 | rouge = proche bord (affichage), Z = champ lisse sur line_b1 + prior cell.1


In [ ]:
# Cellule 3/3 - Triangulation Poisson sur (pcd_0 + nuage source original de mesh_super_1)

# Nuage source original lié à mesh_super_1
if "pcd_super_1" in globals() and isinstance(pcd_super_1, o3d.geometry.PointCloud):
    pcd_src = pcd_super_1
else:
    pcd_src = o3d.geometry.PointCloud()
    pcd_src.points = o3d.utility.Vector3dVector(np.asarray(mesh_super_1.vertices).copy())

# Fusion des deux nuages
pts_src = np.asarray(pcd_src.points, dtype=np.float64)
pts_floor = np.asarray(pcd_0.points, dtype=np.float64)
pts_merge = np.vstack([pts_src, pts_floor])

# Nettoyage de base
finite = np.isfinite(pts_merge).all(axis=1)
pts_merge = pts_merge[finite]
pcd_merge = o3d.geometry.PointCloud()
pcd_merge.points = o3d.utility.Vector3dVector(pts_merge)
pcd_merge = pcd_merge.voxel_down_sample(voxel_size=0.005)

# Normales + Poisson
pcd_merge.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.03, max_nn=30),
    fast_normal_computation=True,
)
pcd_merge.orient_normals_to_align_with_direction([0.0, 0.0, -1.0])

mesh_poisson_0super, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd_merge, depth=15, n_threads=2
)
dens = np.asarray(dens)
if dens.size:
    thr = np.quantile(dens, 0.01)
    mesh_poisson_0super.remove_vertices_by_mask(dens < thr)

mesh_poisson_0super.remove_duplicated_vertices()
mesh_poisson_0super.remove_duplicated_triangles()
mesh_poisson_0super.remove_degenerate_triangles()
mesh_poisson_0super.remove_unreferenced_vertices()
mesh_poisson_0super.compute_triangle_normals()
mesh_poisson_0super.compute_vertex_normals()

print(
    f"[Poisson pcd_0 + src] points={len(pcd_merge.points)} | triangles={len(mesh_poisson_0super.triangles)}"
)

# Affichage
pcd_disp = pcd_merge.voxel_down_sample(voxel_size=0.01)
pcd_disp.paint_uniform_color([0.20, 0.70, 1.00])
mesh_poisson_0super.paint_uniform_color([0.92, 0.38, 0.28])

o3d.visualization.draw_geometries(
    [pcd_disp, mesh_poisson_0super],
    window_name="Poisson — pcd_0 + nuage source mesh_super_1",
    mesh_show_back_face=True,
)

[Poisson pcd_0 + src] points=54507 | triangles=365110


In [ ]:
# Volume (m³) par voxelisation à partir du nuage pcd_augmented_pos
# Échelle inventée : la plus grande dimension du nuage = 1 m

pts = np.asarray(pcd_super_1_ds.points)
min_pt = pts.min(axis=0)
max_pt = pts.max(axis=0)
extent = max_pt - min_pt
max_extent = extent.max()
# Mise à l'échelle : 1 m pour la plus grande dimension
scale_to_m = 1.0 / max_extent if max_extent > 0 else 1.0
pts_m = pts * scale_to_m

pcd_m = o3d.geometry.PointCloud()
pcd_m.points = o3d.utility.Vector3dVector(pts_m)
if pcd_super_1_ds.has_colors():
    pcd_m.colors = pcd_super_1_ds.colors

# Voxelisation : taille de voxel en m (ex. 2 cm)
voxel_size_m = 0.01
voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd_super_1_ds, voxel_size=voxel_size_m)
n_voxels = len(voxel_grid.get_voxels())
volume_m3 = n_voxels * (voxel_size_m ** 3)

print(f"Échelle : {max_extent:.4f} unités → 1 m")
print(f"Voxel size : {voxel_size_m*100:.1f} cm")
print(f"Nombre de voxels occupés : {n_voxels}")
print(f"Volume (voxelisation) : {volume_m3:.4f} m³")

Échelle : 0.6785 unités → 1 m
Voxel size : 1.0 cm
Nombre de voxels occupés : 7655
Volume (voxelisation) : 0.0077 m³


In [ ]:
# Comparaison des méthodes de fermeture (watertight) sur le même maillage Poisson HD
# Prérequis : exécuter la cellule « Poisson watertight » (produit `mesh_poisson_hd`).
# Les variantes sont décalées en X pour inspection visuelle dans une seule fenêtre.

def _clone_triangle_mesh(mesh):
    m = o3d.geometry.TriangleMesh()
    m.vertices = o3d.utility.Vector3dVector(np.asarray(mesh.vertices, dtype=np.float64).copy())
    m.triangles = o3d.utility.Vector3iVector(np.asarray(mesh.triangles, dtype=np.int32).copy())
    m.remove_duplicated_vertices()
    m.remove_duplicated_triangles()
    m.remove_degenerate_triangles()
    m.remove_unreferenced_vertices()
    m.compute_triangle_normals()
    m.compute_vertex_normals()
    return m

def _remove_aberrant_vertices(mesh, threshold=5.0):
    """Supprime les sommets très isolés ou aberrants selon la distance au barycentre/médiane."""
    verts = np.asarray(mesh.vertices)
    centroid = np.median(verts, axis=0)
    dists = np.linalg.norm(verts - centroid, axis=1)
    dist_th = np.median(dists) + threshold * np.std(dists)
    mask = dists < dist_th
    if not np.all(mask):
        # On conserve uniquement les triangles valides avec sommets dans la sélection
        idx_map = -np.ones(len(verts), dtype=np.int32)
        idx_map[mask] = np.arange(np.count_nonzero(mask))
        new_verts = verts[mask]
        triangles = np.asarray(mesh.triangles)
        tri_mask = np.all(mask[triangles], axis=1)
        new_tris = idx_map[triangles[tri_mask]]
        result = o3d.geometry.TriangleMesh()
        result.vertices = o3d.utility.Vector3dVector(new_verts)
        result.triangles = o3d.utility.Vector3iVector(new_tris)
        result.remove_duplicated_vertices()
        result.remove_duplicated_triangles()
        result.remove_degenerate_triangles()
        result.remove_unreferenced_vertices()
        result.compute_triangle_normals()
        result.compute_vertex_normals()
        return result
    return mesh

def _watertight_row(name, mesh, note=""):
    n_b = _count_boundary_edges(mesh) if "_count_boundary_edges" in globals() else -1
    wt = mesh.is_watertight()
    em = mesh.is_edge_manifold()
    vol = None
    err = ""
    if wt:
        try:
            vol = mesh.get_volume()
        except Exception as e:
            err = f"volume: {e}"
    return {
        "méthode": name,
        "tri": len(mesh.triangles),
        "vert": len(mesh.vertices),
        "bord": n_b,
        "watertight": wt,
        "edge_manifold": em,
        "volume": vol,
        "note": note or err,
    }


if "mesh_poisson_hd" not in globals() or len(mesh_poisson_hd.triangles) == 0:
    raise RuntimeError("Exécute d'abord la cellule Poisson watertight (mesh_poisson_hd).")

base = mesh_poisson_hd
rows = []
meshes_vis = []
dx = 1.35 * float(np.max(np.asarray(base.get_axis_aligned_bounding_box().get_extent())))

palette = [
    [0.95, 0.45, 0.35],
    [0.25, 0.55, 0.95],
    [0.35, 0.85, 0.40],
    [0.75, 0.45, 0.90],
    [0.55, 0.55, 0.55],
]

# 1) Poisson HD sans post-traitement watertight
m0 = _clone_triangle_mesh(base)
m0.paint_uniform_color(palette[0])
rows.append(_watertight_row("1 Poisson HD (brut)", m0))
meshes_vis.append(m0)

# 2) Patch petits trous (même idée que mesh_watertight)
m1 = _clone_triangle_mesh(base)
if "_patch_small_holes_from_boundary_loops" in globals():
    m1, n_loops, bb, ba = _patch_small_holes_from_boundary_loops(
        m1, max_loop_edges=22, max_diameter_ratio=0.07
    )
    note = f"loops_remplies={n_loops} bord {bb}->{ba}"
else:
    note = "fonction absente"
m1.compute_triangle_normals()
m1.compute_vertex_normals()
# Nettoyage des points aberrants pour la variante verte
m1_clean = _remove_aberrant_vertices(m1, threshold=5.0)

# Étape supplémentaire : s'assurer que le mesh est watertight. Si ce n'est pas le cas, tente un remplissage tensor.
watertight_note = ""
if not m1_clean.is_watertight():
    if "_seal_mesh_holes_safe" in globals():
        try:
            m1_clean, hbb, hba, hok = _seal_mesh_holes_safe(m1_clean, hole_size=8e4, enable_tensor=True)
            watertight_note = f" | watertight post-tensor={hok} bord {hbb}->{hba}"
        except Exception as e:
            watertight_note = f" | tentative tensor failed: {e}"
    else:
        watertight_note = " | _seal_mesh_holes_safe absent"

m1_clean.paint_uniform_color(palette[1])
rows.append(_watertight_row("2 + patch boucles bord", m1_clean, note + " | nettoyage points aberrants" + watertight_note))
meshes_vis.append(m1_clean)

# 3) Open3D tensor fill_holes (peut être lourd / échouer)
m2 = _clone_triangle_mesh(base)
fill_note = ""
if "_seal_mesh_holes_safe" in globals():
    try:
        m2, hb, ha, ok = _seal_mesh_holes_safe(m2, hole_size=8e4, enable_tensor=True)
        fill_note = f"tensor_ok={ok} bord {hb}->{ha}"
    except Exception as e:
        fill_note = f"exception: {e}"
else:
    fill_note = "_seal_mesh_holes_safe absent"
m2.compute_triangle_normals()
m2.compute_vertex_normals()
# Nettoyage des points aberrants pour la variante mauve
m2_clean = _remove_aberrant_vertices(m2, threshold=5.0)
m2_clean.paint_uniform_color(palette[2])
rows.append(_watertight_row("3 + fill_holes (tensor)", m2_clean, fill_note + " | nettoyage points aberrants"))
meshes_vis.append(m2_clean)

# 4) Enchaînement patch puis tensor
m3 = _clone_triangle_mesh(base)
combo_note = ""
try:
    if "_patch_small_holes_from_boundary_loops" in globals():
        m3, _, _, _ = _patch_small_holes_from_boundary_loops(
            m3, max_loop_edges=22, max_diameter_ratio=0.07
        )
    if "_seal_mesh_holes_safe" in globals():
        m3, hb, ha, ok = _seal_mesh_holes_safe(m3, hole_size=8e4, enable_tensor=True)
        combo_note = f"patch puis tensor | bord {hb}->{ha} ok={ok}"
except Exception as e:
    combo_note = f"exception: {e}"
m3.compute_triangle_normals()
m3.compute_vertex_normals()
m3.paint_uniform_color(palette[3])
rows.append(_watertight_row("4 patch puis tensor", m3, combo_note))
meshes_vis.append(m3)

# 5) Enveloppe convexe (référence : toujours étanche mais géométrie différente)
try:
    hull, _ = base.compute_convex_hull()
    hull = _clone_triangle_mesh(hull)
    hull.paint_uniform_color(palette[4])
    rows.append(_watertight_row("5 Convex hull (réf.)", hull, "ne suit pas la forme"))
    meshes_vis.append(hull)
except Exception as e:
    rows.append(
        {
            "méthode": "5 Convex hull",
            "tri": "-",
            "vert": "-",
            "bord": "-",
            "watertight": "-",
            "edge_manifold": "-",
            "volume": None,
            "note": str(e),
        }
    )

# Tableau récapitulatif
hdr = f"{'méthode':<28} {'tri':>7} {'vert':>7} {'bord':>6} {'WT':>5} {'manif':>5} {'volume':>12}  note"
print(hdr)
print("-" * len(hdr))
for r in rows:
    vol_s = f"{r['volume']:.6g}" if r["volume"] is not None else "-"
    print(
        f"{r['méthode']:<28} {str(r['tri']):>7} {str(r['vert']):>7} {str(r['bord']):>6} "
        f"{str(r['watertight']):>5} {str(r['edge_manifold']):>5} {vol_s:>12}  {r['note']}"
    )

# Superposition décalée : gauche = brut, droite = variantes
for i, m in enumerate(meshes_vis):
    m.translate((i * dx, 0.0, 0.0))

o3d.visualization.draw_geometries(
    meshes_vis,
    window_name="Comparaison watertight (décalage X)",
    mesh_show_back_face=True,
)


[Open3D WARNING] Ignoring attribute 'normals' for TensorMap with primary key 'indices'
[Open3D WARNING] Ignoring attribute 'normals' for TensorMap with primary key 'indices'
[Open3D WARNING] Ignoring attribute 'normals' for TensorMap with primary key 'indices'
méthode                          tri    vert   bord    WT manif       volume  note
----------------------------------------------------------------------------------
1 Poisson HD (brut)            26533   13308    145 False False            -  
2 + patch boucles bord         26651   13307     27 False  True            -  loops_remplies=0 bord 145->145 | nettoyage points aberrants | watertight post-tensor=True bord 145->27
3 + fill_holes (tensor)        26648   13306     30 False  True            -  tensor_ok=True bord 145->27 | nettoyage points aberrants
4 patch puis tensor            26651   13307     27 False  True            -  patch puis tensor | bord 145->27 ok=True
5 Convex hull (réf.)             200     102      0  True  

In [ ]:

if 'pcd_combo' in globals() and isinstance(pcd_combo, o3d.geometry.PointCloud):

    def _crust_mesh_from_point_cloud(
        pcd,
        k_nn=20,
        pole_scale=2.0,
        max_points=9000,
        rng_seed=0,
    ):
        """CRUST simplifié (esprit Amenta–Bern) : normales par PCA locale, pôles ±rho*n,
        2e triangulation Delaunay 3D, faces dont les 3 sommets sont des échantillons d'origine.
        Open3D n'a pas de CRUST natif ; ce n'est pas le filtre « sphère vide » complet.
        Dépend de scipy (déjà présent avec geopandas/tensorflow dans ce projet)."""
        try:
            from scipy.spatial import Delaunay, cKDTree
        except ImportError:
            print("[CRUST] scipy introuvable — pip install scipy")
            return o3d.geometry.TriangleMesh()

        P = np.asarray(pcd.points, dtype=np.float64)
        n_all = P.shape[0]
        if n_all < 8:
            return o3d.geometry.TriangleMesh()
        if n_all > max_points:
            rng = np.random.default_rng(rng_seed)
            sel = rng.choice(n_all, size=max_points, replace=False)
            P = P[sel]
        n = P.shape[0]
        tree = cKDTree(P)
        kn = int(min(max(4, k_nn), n - 1))
        N = np.zeros_like(P)
        dist_med = np.zeros(n)
        for i in range(n):
            dists, jj = tree.query(P[i], k=kn + 1)
            jj = np.atleast_1d(jj)
            neigh = P[jj[1:]]
            dist_med[i] = float(np.median(np.asarray(dists[1:], dtype=np.float64)))
            c = neigh.mean(axis=0)
            X = neigh - c
            _, _, vt = np.linalg.svd(X, full_matrices=False)
            nrm = vt[-1]
            if np.dot(nrm, P[i] - c) < 0:
                nrm = -nrm
            nn = np.linalg.norm(nrm)
            N[i] = nrm / (nn + 1e-12)
        rho = float(np.median(dist_med)) * float(pole_scale)
        Q = np.vstack([P, P + rho * N, P - rho * N])
        tri = Delaunay(Q)
        faces = set()
        for tet in tri.simplices:
            a, b, c, d = (int(tet[0]), int(tet[1]), int(tet[2]), int(tet[3]))
            for (i, j, k) in ((a, b, c), (a, b, d), (a, c, d), (b, c, d)):
                if i < n and j < n and k < n:
                    faces.add(tuple(sorted((i, j, k))))
        if not faces:
            return o3d.geometry.TriangleMesh()
        tris = np.array(list(faces), dtype=np.int32)
        mesh = o3d.geometry.TriangleMesh()
        mesh.vertices = o3d.utility.Vector3dVector(P)
        mesh.triangles = o3d.utility.Vector3iVector(tris)
        mesh.remove_duplicated_triangles()
        mesh.remove_duplicated_vertices()
        mesh.remove_degenerate_triangles()
        mesh.remove_unreferenced_vertices()
        lbl, cnt, _ = mesh.cluster_connected_triangles()
        cnt = np.asarray(cnt)
        if cnt.size > 0:
            keep = int(np.argmax(cnt))
            mesh.remove_triangles_by_mask(np.asarray(lbl) != keep)
            mesh.remove_unreferenced_vertices()
        mesh.compute_vertex_normals()
        return mesh

    # 1. Downsample le nuage de points source
    voxel_size = 0.001  # Ajuste la taille de voxel si nécessaire
    pcd_source_display = pcd_combo.voxel_down_sample(voxel_size=voxel_size)
    pcd_source_display.paint_uniform_color([0.20, 0.70, 1.00])

    # 2. Triangulation de Poisson
    pcd_for_poisson = o3d.geometry.PointCloud(pcd_source_display)  # copié pour Poisson
    pcd_for_poisson.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
    poisson_mesh, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        pcd_for_poisson, depth=10
    )
    poisson_mesh.compute_vertex_normals()
    poisson_mesh.paint_uniform_color([0.7, 0.9, 0.6])

    # 3. Triangulation Alpha Shape
    alpha = 0.03  # à ajuster selon densité/échelle
    alpha_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_alpha_shape(
        pcd_source_display, alpha
    )
    alpha_mesh.compute_vertex_normals()
    alpha_mesh.paint_uniform_color([0.98, 0.77, 0.25])

    # 4. Triangulation Ball Pivoting
    # Pour ball pivoting, il faut des normales et un radius adapté à la densité et l'échelle du nuage.
    pcd_for_bp = o3d.geometry.PointCloud(pcd_source_display)
    pcd_for_bp.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
    # Choisir plusieurs rayons pour combler les trous, en fonction de la densité du nuage
    distances = pcd_for_bp.compute_nearest_neighbor_distance()
    avg_dist = np.mean(distances)
    radii = [avg_dist * 1.0, avg_dist * 1.5, avg_dist * 2.0]  # augmenter si trous intérieurs
    bp_mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
        pcd_for_bp, o3d.utility.DoubleVector(radii)
    )
    bp_mesh.compute_vertex_normals()
    bp_mesh.paint_uniform_color([1.0, 0.5, 0.5])

    # 5. Triangulation CRUST (scipy : Delaunay sur échantillons + pôles estimés)
    crust_mesh = _crust_mesh_from_point_cloud(
        pcd_source_display, k_nn=30, pole_scale=4.0, max_points=90000
    )

    # Colmatage modéré (moins agressif que _seal_mesh_holes_safe : pas de remove_non_manifold,
    # petites boucles de bord puis fill_holes avec surface de trou plafonnée).
   

    crust_mesh.paint_uniform_color([0.55, 0.35, 0.95])

    # Affichage côte à côte : nuage, Poisson, Alpha, BPA, CRUST
    dx = 1  # Espace entre les objets
    disp_cloud = pcd_source_display.translate((0, 0, 0))
    disp_poisson = poisson_mesh.translate((dx, 0, 0))
    disp_alpha = alpha_mesh.translate((2 * dx, 0, 0))
    disp_bp = bp_mesh.translate((3 * dx, 0, 0))
    disp_crust = crust_mesh.translate((4 * dx, 0, 0))

    o3d.visualization.draw_geometries(
        [disp_cloud, disp_poisson, disp_alpha, disp_bp, disp_crust],
        window_name="Nuage (ds), Poisson, AlphaShape, BallPivoting, CRUST",
        mesh_show_back_face=True
    )
else:
    print("pcd_combo absent — exécuter la cellule qui construit pcd_combo.")

In [ ]:
# Recadrer disp_poisson : supprime les triangles hors du AABB du nuage original (pcd_combo).
# Prérequis : cellule comparaison exécutée (disp_poisson, dx, pcd_combo).
# Le maillage d'affichage est décalé de (dx,0,0) : la boîte est translatée de la même façon.

if not (
    "disp_poisson" in globals()
    and "pcd_combo" in globals()
    and isinstance(disp_poisson, o3d.geometry.TriangleMesh)
    and isinstance(pcd_combo, o3d.geometry.PointCloud)
):
    raise RuntimeError(
        "Exécute d'abord la cellule qui crée disp_poisson et pcd_combo (comparaison Poisson / …)."
    )

_dx = float(dx) if "dx" in globals() else 1.0
_shift = np.array([_dx, 0.0, 0.0], dtype=np.float64)

# Marge relative sur le AABB source (0 = strictement l'enveloppe du nuage ; >0 élargit un peu)
bbox_margin_ratio = 0.0

_aabb0 = pcd_combo.get_axis_aligned_bounding_box()
_ext = np.asarray(_aabb0.get_extent(), dtype=np.float64)
_pad = (bbox_margin_ratio * _ext)/100
_min = np.asarray(_aabb0.min_bound, dtype=np.float64) - _pad
_max = np.asarray(_aabb0.max_bound, dtype=np.float64) + _pad
_aabb_disp = o3d.geometry.AxisAlignedBoundingBox(_min + _shift, _max + _shift)

_n_tri0 = len(disp_poisson.triangles)
disp_poisson = disp_poisson.crop(_aabb_disp)
disp_poisson.remove_duplicated_vertices()
disp_poisson.remove_duplicated_triangles()
disp_poisson.remove_degenerate_triangles()
disp_poisson.remove_unreferenced_vertices()
disp_poisson.compute_vertex_normals()
_n_tri1 = len(disp_poisson.triangles)
print(
    f"[Crop Poisson] AABB nuage source (décalé dx={_dx:g}) | "
    f"triangles {_n_tri0} -> {_n_tri1}"
)
o3d.visualization.draw_geometries(
        [disp_cloud,disp_poisson],
        window_name="Nuage (ds), Poisson, AlphaShape, BallPivoting, CRUST",
        mesh_show_back_face=True
    )

[Crop Poisson] AABB nuage source (décalé dx=0.992866) | triangles 0 -> 0
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.


In [ ]:
# Volume « contenu » par disp_poisson (maillage triangulaire)
# Prérequis : disp_poisson défini (cellule comparaison ; éventuellement recadrage).
# Unités = unités du maillage (mêmes que depth / nuage, sauf mise à l'échelle métrique ailleurs).

if "disp_poisson" not in globals() or not isinstance(disp_poisson, o3d.geometry.TriangleMesh):
    raise RuntimeError("disp_poisson introuvable — exécuter la cellule de comparaison d'abord.")

_m = disp_poisson
print(
    f"[Volume disp_poisson] triangles={len(_m.triangles)} | "
    f"watertight={_m.is_watertight()} | edge_manifold={_m.is_edge_manifold()}"
)

vol = None
methode = None

if _m.is_watertight():
    try:
        vol = float(np.abs(_m.get_volume()))
        methode = "get_volume() (maillage étanche)"
    except Exception as e:
        print(f"[Volume] get_volume indisponible : {e}")

if vol is None:
    try:
        _hull, _ = _m.compute_convex_hull()
        if _hull.is_watertight():
            vol = float(np.abs(_hull.get_volume()))
            methode = "enveloppe convexe des sommets (sur-estimation si forme non convexe)"
    except Exception as e:
        print(f"[Volume] convex_hull indisponible : {e}")

if vol is not None:
    print(f"[Volume disp_poisson] {vol:.8g} (unités³) — {methode}")
else:
    print("[Volume disp_poisson] impossible à estimer automatiquement (essayez watertight / fill_holes).")


[Volume disp_poisson] triangles=178585 | watertight=False | edge_manifold=False
[Volume disp_poisson] 0.11766204 (unités³) — enveloppe convexe des sommets (sur-estimation si forme non convexe)
